# 17 — PyIngestKit Integration

**Related guide:** [24_PYINGESTKIT_INTEGRATION.md](../docs/guides/24_PYINGESTKIT_INTEGRATION.md)

**Related script:** [integrations/pyingestkit/atomic_job.py](../examples/integrations/pyingestkit/atomic_job.py)

This notebook follows the DX05 loop: **Concept → Construction → Inspection → Modification → Execution → Observation → Experiment**.

## Construction

Use the specialized anti-corruption adapter while keeping one PyIngestKit job atomic to PyWorkflowKit.

In [ ]:
from pyworkflowkit._compat.v1_root import TaskHandle, WorkflowRuntime, workflow
from pyworkflowkit.ecosystem import RunContext
from pyworkflowkit.integrations.pyingestkit import PyIngestKitRunResult, pyingestkit_task


class ExamplePyIngestKitJob:
    def run(self, *, context: RunContext) -> PyIngestKitRunResult:
        return PyIngestKitRunResult(
            external_run_id=f"ingest-{context.task_run_id}",
            succeeded=True,
            output={"rows": 3},
        )


ingest = pyingestkit_task(
    id="ingest",
    job_ref="daily-customers",
    job=ExamplePyIngestKitJob(),
)


@workflow(id="notebooks.pyingestkit", version="1")
def demo() -> tuple[TaskHandle, ...]:
    return (ingest,)

In [ ]:
runtime = WorkflowRuntime()
runtime.register(ingest.handler_ref, ingest.handler)
definition = demo.build()
run = runtime.run(definition)
lineage = runtime.lineage(definition, run.run_id)
external_refs = sum(len(task.external_ref_ids) for task in lineage.tasks)
print({"status": run.status.value, "external_refs": external_refs})

## Experiment

Replace the mock wrapper with a real PyIngestKit adapter in an integration environment without changing workflow semantics.